# 🤖 Model Training & Comparison
Training 3 models and comparing on business metrics.

In [ ]:
from src.data.loader import load_raw_data
from src.data.preprocessor import clean_data, get_features_and_target
from src.features.engineering import engineer_features
from src.models.baseline import train_baseline
from src.models.random_forest import train_random_forest
from src.models.xgboost_model import train_xgboost
import pandas as pd

df = clean_data(load_raw_data())
df = engineer_features(df)
X, y = get_features_and_target(df)
print(f"Dataset: {X.shape[0]:,} customers, {X.shape[1]} features")
print(f"Churn rate: {y.mean():.1%}")

In [ ]:
# Train all 3 models
print('Training Logistic Regression...')
lr_model, X_test, y_test, lr_results = train_baseline(X, y)
print('\nTraining Random Forest...')
rf_model, _, _, rf_results = train_random_forest(X, y)
print('\nTraining XGBoost...')
xgb_model, _, _, xgb_results = train_xgboost(X, y)

In [ ]:
# Compare all models
results = pd.DataFrame([lr_results, rf_results, xgb_results])
cols = ['model','f1','auc','precision','recall','total_cost_ils','saved_revenue_ils']
results[cols].set_index('model').round(4)

In [ ]:
# Business cost comparison
import matplotlib.pyplot as plt
models = [lr_results['model'], rf_results['model'], xgb_results['model']]
costs = [lr_results['total_cost_ils'], rf_results['total_cost_ils'], xgb_results['total_cost_ils']]
revenue = [lr_results['saved_revenue_ils'], rf_results['saved_revenue_ils'], xgb_results['saved_revenue_ils']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(models, costs, color=['#3498db','#e67e22','#e74c3c'])
axes[0].set_title('Total Error Cost (₪) — lower is better')
axes[1].bar(models, revenue, color=['#3498db','#e67e22','#2ecc71'])
axes[1].set_title('Saved Revenue (₪) — higher is better')
plt.tight_layout()
plt.show()

In [ ]:
# Winner
print('🏆 Champion Model: XGBoost (tuned)')
print(f"F1: {xgb_results['f1']}")
print(f"AUC: {xgb_results['auc']}")
print(f"Recall: {xgb_results['recall']}")
print(f"Saved Revenue: ₪{xgb_results['saved_revenue_ils']:,}")